# 01 — Dataset Exploration

This notebook explores all four industrial datasets included in this repository:
- **NASA CMAPSS** — Turbofan engine run-to-failure
- **IMS Bearing** — Rolling element bearing run-to-failure
- **Paderborn Bearing** — Bearing fault classification
- **CWRU Bearing** — Benchmark bearing fault dataset

We visualise raw signals, sensor channel statistics, and degradation trends.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

## 1. NASA CMAPSS Turbofan Dataset

In [ ]:
from datasets.cmapss_loader import CMAPSSLoader

# Load FD001 subset
loader = CMAPSSLoader(
    subset='FD001',
    data_dir='../datasets/data',
    max_rul=125,
    window_size=30,
    normalize=True
)

try:
    X_train, y_train, X_test, y_test = loader.load()
    print(f'X_train shape: {X_train.shape}  (N, seq_len, n_features)')
    print(f'y_train shape: {y_train.shape}  (N,) — RUL labels')
    print(f'X_test  shape: {X_test.shape}   (N_units, seq_len, n_features)')
    print(f'y_test  shape: {y_test.shape}   (N_units,) — ground-truth RUL')
    print(f'\nRUL range — train: [{y_train.min():.0f}, {y_train.max():.0f}]')
    print(f'RUL range — test:  [{y_test.min():.0f}, {y_test.max():.0f}]')
    cmapss_loaded = True
except FileNotFoundError as e:
    print(f'Data not found: {e}')
    print('Generating synthetic data for demonstration...')
    np.random.seed(42)
    X_train = np.random.randn(17000, 30, 14).astype(np.float32)
    y_train = np.random.uniform(0, 125, 17000).astype(np.float32)
    X_test  = np.random.randn(100, 30, 14).astype(np.float32)
    y_test  = np.random.uniform(0, 125, 100).astype(np.float32)
    cmapss_loaded = False

In [ ]:
# RUL label distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(y_train, bins=50, color='steelblue', edgecolor='white', alpha=0.85)
axes[0].set_xlabel('RUL (cycles)')
axes[0].set_ylabel('Count')
axes[0].set_title('Training RUL Label Distribution (FD001)')
axes[0].axvline(125, color='red', linestyle='--', label='Max RUL cap (125)')
axes[0].legend()

axes[1].hist(y_test, bins=20, color='coral', edgecolor='white', alpha=0.85)
axes[1].set_xlabel('RUL (cycles)')
axes[1].set_ylabel('Count')
axes[1].set_title('Test Ground-Truth RUL Distribution (FD001)')

plt.tight_layout()
plt.show()

In [ ]:
# Sensor channel heatmap (mean value across training windows)
sensor_means = X_train.mean(axis=(0, 1))   # (n_features,)
sensor_stds  = X_train.std(axis=(0, 1))

feature_names = loader.feature_names if cmapss_loaded else [f'sensor_{i:02d}' for i in range(14)]

fig, ax = plt.subplots(figsize=(12, 3))
x = np.arange(len(feature_names))
ax.bar(x, sensor_means, yerr=sensor_stds, capsize=3, color='steelblue', alpha=0.8)
ax.set_xticks(x)
ax.set_xticklabels(feature_names, rotation=45, ha='right')
ax.set_ylabel('Mean ± Std (normalised)')
ax.set_title('Sensor Channel Statistics — Training Data (CMAPSS FD001)')
plt.tight_layout()
plt.show()

In [ ]:
# Sensor correlation matrix
import seaborn as sns

# Use last timestep of each window as a representative sample
X_last = X_train[:, -1, :]   # (N, n_features)
corr = np.corrcoef(X_last.T)

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(
    corr,
    mask=mask,
    annot=True,
    fmt='.1f',
    cmap='RdBu_r',
    vmin=-1, vmax=1,
    xticklabels=feature_names,
    yticklabels=feature_names,
    ax=ax,
    annot_kws={'size': 7}
)
ax.set_title('Sensor Correlation Matrix (CMAPSS FD001)')
plt.tight_layout()
plt.show()

## 2. Degradation Trends

In [ ]:
# Show how RUL decreases over cycles for a sample of training units
# We reconstruct the trajectory from the y_train labels

np.random.seed(0)

# Synthetic degradation trajectories for illustration
n_units = 5
fig, ax = plt.subplots(figsize=(12, 5))

colors = plt.cm.viridis(np.linspace(0.1, 0.9, n_units))
for i in range(n_units):
    life = np.random.randint(100, 250)
    cycles = np.arange(life)
    rul = np.maximum(0, np.minimum(125, life - cycles - 1))
    ax.plot(cycles, rul, color=colors[i], alpha=0.7, label=f'Unit {i+1} (life={life})')

ax.axhline(y=125, color='red', linestyle='--', linewidth=1.5, label='Max RUL cap (125)')
ax.set_xlabel('Operating Cycle')
ax.set_ylabel('Remaining Useful Life (cycles)')
ax.set_title('Piecewise-Linear RUL Labels (CMAPSS style)')
ax.legend(loc='upper right', fontsize=9)
plt.tight_layout()
plt.show()

## 3. IMS Bearing Dataset

In [ ]:
from datasets.ims_loader import IMSLoader

ims_loader = IMSLoader(data_dir='../datasets/data/IMS', run=1, window_size=1024)

try:
    X_ims, health_idx, timestamps = ims_loader.load()
    print(f'Loaded {len(X_ims)} files, window_size=1024, channels={X_ims.shape[-1]}')
    ims_loaded = True
except FileNotFoundError as e:
    print(f'IMS data not found: {e}')
    # Generate synthetic degradation
    n = 984
    health_idx = np.zeros(n)
    health_idx[700:] = np.linspace(0, 1, n - 700) ** 2
    X_ims = np.random.randn(n, 1024, 4).astype(np.float32) * (1 + health_idx[:, None, None] * 5)
    timestamps = [str(i) for i in range(n)]
    ims_loaded = False

In [ ]:
# Health index timeline
fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

axes[0].plot(health_idx, color='steelblue', linewidth=0.8)
axes[0].set_ylabel('Health Index (RMS-based)')
axes[0].set_title('IMS Bearing Run 1 — Health Index Timeline')
axes[0].fill_between(range(len(health_idx)), health_idx, alpha=0.2, color='steelblue')

# RMS of first bearing channel over time
rms_over_time = np.sqrt(np.mean(X_ims[:, :, 0] ** 2, axis=1))
axes[1].plot(rms_over_time, color='coral', linewidth=0.8)
axes[1].set_ylabel('RMS Vibration (Bearing 1)')
axes[1].set_xlabel('File index (time)')

plt.tight_layout()
plt.show()

## 4. CWRU Bearing Dataset

In [ ]:
from datasets.cwru_loader import CWRULoader

cwru_loader = CWRULoader(
    data_dir='../datasets/data/CWRU',
    fault_sizes=['normal', '0.007', '0.014', '0.021'],
    window_size=1024,
    stride=512
)

try:
    X_cwru, y_cwru, label_names = cwru_loader.load()
    print(f'Loaded: {X_cwru.shape}  |  Labels: {np.bincount(y_cwru)}')
    cwru_loaded = True
except FileNotFoundError as e:
    print(f'CWRU data not found: {e}')
    # Generate 4-class synthetic data
    np.random.seed(42)
    n_per_class = 500
    X_cwru = np.vstack([
        np.random.randn(n_per_class, 1024),            # normal
        np.random.randn(n_per_class, 1024) * 1.5 + np.sin(np.linspace(0, 10*np.pi, 1024)),  # IR
        np.random.randn(n_per_class, 1024) * 1.3,     # ball
        np.random.randn(n_per_class, 1024) * 1.8,     # OR
    ]).astype(np.float32)
    y_cwru = np.repeat([0, 1, 2, 3], n_per_class)
    label_names = ['normal', 'inner_race', 'ball', 'outer_race']
    cwru_loaded = False

In [ ]:
# Visualise sample signals for each class
fig, axes = plt.subplots(len(label_names), 1, figsize=(14, 9), sharex=True)
n_show = min(512, X_cwru.shape[1])

colors = ['steelblue', 'coral', 'forestgreen', 'mediumpurple']

for i, (name, color) in enumerate(zip(label_names, colors)):
    mask = y_cwru == i
    if mask.sum() == 0:
        continue
    sample = X_cwru[mask][0, :n_show]
    axes[i].plot(sample, color=color, linewidth=0.6)
    axes[i].set_ylabel(name.replace('_', ' ').title(), fontsize=9)
    axes[i].set_ylim(-4, 4)

axes[-1].set_xlabel('Sample index')
fig.suptitle('CWRU Bearing — Raw Vibration Signals by Fault Type', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Class distribution
fig, ax = plt.subplots(figsize=(7, 4))
counts = np.bincount(y_cwru, minlength=len(label_names))
bars = ax.bar(label_names, counts, color=colors[:len(label_names)], alpha=0.85, edgecolor='white')
for bar, count in zip(bars, counts):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
            str(count), ha='center', va='bottom', fontsize=10)
ax.set_xlabel('Fault Type')
ax.set_ylabel('Number of Windows')
ax.set_title('CWRU Dataset — Class Distribution')
plt.tight_layout()
plt.show()

## Summary

| Dataset | Task | Samples | Features | Classes/Target |
|---------|------|---------|----------|----------------|
| CMAPSS FD001 | RUL Regression | ~17K windows | 14 sensors | RUL (0–125 cycles) |
| IMS Bearing | Anomaly / RUL | ~984 files | 4 channels | Health index |
| CWRU | Fault Classification | ~2K windows | 1024 samples | 4 classes |
| Paderborn | Fault Classification | Variable | 4096 samples | 4 classes |

**Next:** [02 — Feature Engineering](02_feature_engineering.ipynb)